In [ ]:
# %% [markdown]
# # Retrosynthese-Challenge: Product-SMILES → Reactant-SMILES
#
# Aufgabe:
#
# ```text
# data_train.csv:
# Reactants >> Products
# ```
#
# Ziel:
#
# ```text
# Product SMILES → Reactant SMILES
# ```
#
# Dieses Notebook verwendet eine verbesserte Baseline:
#
# 1. RDKit-Canonicalisierung passend zur Evaluation
# 2. Exact Product Lookup
# 3. Optionaler No-Stereo Lookup
# 4. Top-k Fingerprint Nearest Neighbor mit gewichteter Abstimmung
# 5. lokale Validierung
# 6. automatische Parameter-Suche
# 7. finale Submission-Erzeugung

# %% [markdown]
# ## Zelle 1: Setup
#
# Lege dieses Notebook in denselben Ordner wie:
#
# ```text
# data_train.csv
# product_smiles_test.csv
# sample_submission.csv
# top1_accuracy.py
# ```
#
# Falls RDKit fehlt:
#
# ```bash
# conda install -c conda-forge rdkit
# ```
#
# oder:
#
# ```bash
# pip install rdkit-pypi
# ```

# %%
from pathlib import Path
from collections import Counter, defaultdict
import random

import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

DATA_DIR = Path(".")

TRAIN_PATH = DATA_DIR / "data_train.csv"
TEST_PATH = DATA_DIR / "product_smiles_test.csv"
SAMPLE_SUB_PATH = DATA_DIR / "sample_submission.csv"

print("Working directory:", Path.cwd())
print("Train exists:", TRAIN_PATH.exists())
print("Test exists:", TEST_PATH.exists())
print("Sample submission exists:", SAMPLE_SUB_PATH.exists())

# %% [markdown]
# ## Zelle 2: Canonicalisierung
#
# Das Evaluationsskript macht pro Molekül:
#
# ```python
# Chem.MolFromSmiles(...)
# Chem.MolToSmiles(...)
# ```
#
# Danach werden die Reactants als Set verglichen.
#
# Wichtig:
#
# - Reihenfolge der Moleküle ist egal.
# - Ungültige SMILES werden ignoriert.
# - Stereochemie kann relevant sein.
# - Für den optionalen Lookup ohne Stereochemie brauchen wir zusätzlich `isomeric=False`.

# %%
def canonical_mol(smiles: str, isomeric: bool = True):
    """
    Canonicalisiert ein einzelnes Molekül-SMILES mit RDKit.
    
    isomeric=True:
        Stereochemie wird erhalten.
    
    isomeric=False:
        Stereochemie wird ignoriert.
    """
    if smiles is None:
        return None

    smiles = str(smiles).strip()

    if smiles == "":
        return None

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    return Chem.MolToSmiles(mol, isomericSmiles=isomeric)


def canonical_set(smiles: str, isomeric: bool = True) -> str:
    """
    Canonicalisiert eine mit '.' getrennte Liste von Molekülen.
    
    Wie in der Evaluation:
    - ungültige Moleküle werden ignoriert
    - Moleküle werden als Set betrachtet
    - Reihenfolge wird stabil sortiert
    """
    if smiles is None:
        return ""

    parts = str(smiles).strip().split(".")
    canon = []

    for part in parts:
        c = canonical_mol(part, isomeric=isomeric)
        if c is not None:
            canon.append(c)

    return ".".join(sorted(set(canon)))


def parse_reaction_line(line: str):
    """
    Parst eine Zeile:
    
    Reactants >> Products
    
    Rückgabe:
    canonical_reactants, canonical_products
    """
    line = str(line).strip()

    if ">>" not in line:
        return None, None

    reactants, products = line.split(">>", 1)

    reactants = canonical_set(reactants.strip(), isomeric=True)
    products = canonical_set(products.strip(), isomeric=True)

    if reactants == "" or products == "":
        return None, None

    return reactants, products

# %% [markdown]
# ## Zelle 3: Lokale Evaluation passend zu `top1_accuracy.py`
#
# Diese Funktion bildet die Logik aus deinem Evaluationsskript nach.
#
# Der wichtige Unterschied zu einer normalen String-Accuracy:
#
# ```text
# O.CC
# ```
#
# und
#
# ```text
# CC.O
# ```
#
# zählen als gleich.

# %%
def process_chemicals_like_evaluation(chemicals):
    """
    Entspricht der Logik aus top1_accuracy.py:
    - jedes Molekül einzeln parsen
    - valide Moleküle canonicalisieren
    - ungültige Moleküle ignorieren
    """
    processed_chemicals = []

    for chem in chemicals:
        m = Chem.MolFromSmiles(str(chem).strip())
        if m is not None:
            processed_chemicals.append(Chem.MolToSmiles(m))

    return processed_chemicals


def calculate_top1_accuracy_from_lists(predictions, true_answers):
    assert len(predictions) == len(true_answers), "Predictions and targets must have same length."

    correct = 0
    total = 0

    for pred, true in zip(predictions, true_answers):
        if pred is None or true is None:
            continue

        pred = str(pred).strip()
        true = str(true).strip()

        if pred == "" or true == "":
            continue

        true_set = set(process_chemicals_like_evaluation(true.split(".")))
        pred_set = set(process_chemicals_like_evaluation(pred.split(".")))

        if true_set == pred_set:
            correct += 1

        total += 1

    if total == 0:
        return 0.0

    return correct / total


def evaluation_equal(pred: str, true: str) -> bool:
    """
    Einzelvergleich nach offizieller Evaluation.
    """
    return calculate_top1_accuracy_from_lists([pred], [true]) == 1.0


# Mini-Tests

# Reihenfolge ist egal
assert calculate_top1_accuracy_from_lists(["CC.O"], ["O.CC"]) == 1.0

# Unterschiedliche valide Moleküle sind falsch
assert calculate_top1_accuracy_from_lists(["CC"], ["CCC"]) == 0.0

# Ungültige SMILES werden ignoriert.
# Deshalb sind beide Seiten hier effektiv leer und gelten als gleich.
assert calculate_top1_accuracy_from_lists(["A.B"], ["B.A"]) == 1.0

print("Evaluation helper works.")

# %% [markdown]
# ## Zelle 4: Dateien laden
#
# `data_train.csv` enthält:
#
# ```text
# Reactants >> Products
# ```
#
# `product_smiles_test.csv` enthält:
#
# ```text
# Product SMILES
# ```
#
# `sample_submission.csv` zeigt das Ziel-Format:
#
# ```text
# Predicted Reactants
# ```

# %%
def read_single_column_text_file(path: Path):
    """
    Liest eine headerlose, einspaltige Datei als Liste nicht-leerer Zeilen.
    """
    with open(path, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f if line.strip() != ""]

    return lines


train_lines = read_single_column_text_file(TRAIN_PATH)
test_products_raw = read_single_column_text_file(TEST_PATH)

print("Number of train reactions:", len(train_lines))
print("Number of test products:", len(test_products_raw))

print()
print("First 4 train lines:")
for x in train_lines[:4]:
    print(x)

print()
print("First 4 test products:")
for x in test_products_raw[:4]:
    print(x)

# %% [markdown]
# ## Zelle 5: Trainingsdaten umdrehen
#
# Aus:
#
# ```text
# Reactants >> Products
# ```
#
# wird:
#
# ```text
# Product → Reactants
# ```

# %%
pairs = []
invalid_lines = 0

for line in train_lines:
    reactants, products = parse_reaction_line(line)

    if reactants is None or products is None:
        invalid_lines += 1
        continue

    pairs.append(
        {
            "product": products,
            "reactants": reactants,
            "raw_line": line,
        }
    )

df = pd.DataFrame(pairs)

print("Valid parsed reactions:", len(df))
print("Invalid lines skipped:", invalid_lines)

df.head()

# %%
print("Unique canonical products:", df["product"].nunique())
print("Unique canonical reactant sets:", df["reactants"].nunique())

duplicate_product_counts = df["product"].value_counts()

print("Products appearing more than once:", (duplicate_product_counts > 1).sum())

df.head(10)

# %% [markdown]
# ## Zelle 6: Verbesserte Modellklasse
#
# Diese Klasse ersetzt die alte `RetrosynthesisBaseline`.
#
# Verbesserungen:
#
# - Exact Lookup mit Stereochemie
# - optionaler Lookup ohne Stereochemie
# - Top-k statt nur Top-1 Nachbar
# - gewichtetes Voting nach Tanimoto-Similarity
# - häufige Trainingsreaktionen bekommen automatisch mehr Gewicht

# %%
class RetrosynthesisTopKBaseline:
    def __init__(
        self,
        radius=2,
        n_bits=2048,
        top_k=20,
        similarity_power=2.0,
        use_no_stereo_lookup=True,
    ):
        self.radius = radius
        self.n_bits = n_bits
        self.top_k = top_k
        self.similarity_power = similarity_power
        self.use_no_stereo_lookup = use_no_stereo_lookup

        self.exact_lookup = defaultdict(Counter)
        self.no_stereo_lookup = defaultdict(Counter)

        self.product_to_best_reactants = {}
        self.no_stereo_to_best_reactants = {}

        self.most_common_reactants = None

        self.train_products = []
        self.train_reactants = []
        self.train_fps = []

    def _fp(self, smiles: str):
        mol = Chem.MolFromSmiles(smiles)

        if mol is None:
            return None

        return AllChem.GetMorganFingerprintAsBitVect(
            mol,
            radius=self.radius,
            nBits=self.n_bits,
        )

    def fit(self, train_df: pd.DataFrame):
        """
        Erwartet ein DataFrame mit Spalten:
        - product
        - reactants
        """
        self.exact_lookup = defaultdict(Counter)
        self.no_stereo_lookup = defaultdict(Counter)

        for product, reactants in zip(train_df["product"], train_df["reactants"]):
            product_full = canonical_set(product, isomeric=True)
            product_no_stereo = canonical_set(product, isomeric=False)

            self.exact_lookup[product_full][reactants] += 1
            self.no_stereo_lookup[product_no_stereo][reactants] += 1

        self.product_to_best_reactants = {
            product: counter.most_common(1)[0][0]
            for product, counter in self.exact_lookup.items()
        }

        self.no_stereo_to_best_reactants = {
            product: counter.most_common(1)[0][0]
            for product, counter in self.no_stereo_lookup.items()
        }

        self.most_common_reactants = Counter(train_df["reactants"]).most_common(1)[0][0]

        self.train_products = []
        self.train_reactants = []
        self.train_fps = []

        # Hier nehmen wir bewusst alle Trainingszeilen, nicht nur eindeutige Products.
        # Dadurch wirken häufige Reaktionen stärker im Voting.
        for product, reactants in zip(train_df["product"], train_df["reactants"]):
            fp = self._fp(product)

            if fp is not None:
                self.train_products.append(product)
                self.train_reactants.append(reactants)
                self.train_fps.append(fp)

        print("Fitted Top-k model")
        print("Training rows with valid fingerprints:", len(self.train_fps))
        print("Exact lookup products:", len(self.product_to_best_reactants))
        print("No-stereo lookup products:", len(self.no_stereo_to_best_reactants))
        print("Most common reactants:", self.most_common_reactants)

        return self

    def predict_one(self, product_smiles: str):
        product_full = canonical_set(product_smiles, isomeric=True)
        product_no_stereo = canonical_set(product_smiles, isomeric=False)

        # 1. Exakter Lookup mit Stereochemie
        if product_full in self.product_to_best_reactants:
            return self.product_to_best_reactants[product_full]

        # 2. Relaxed Lookup ohne Stereochemie
        if self.use_no_stereo_lookup and product_no_stereo in self.no_stereo_to_best_reactants:
            return self.no_stereo_to_best_reactants[product_no_stereo]

        # 3. Top-k Fingerprint Voting
        fp = self._fp(product_full)

        if fp is None or len(self.train_fps) == 0:
            return self.most_common_reactants

        sims = np.array(DataStructs.BulkTanimotoSimilarity(fp, self.train_fps))

        k = min(self.top_k, len(sims))

        top_indices = np.argpartition(-sims, kth=k - 1)[:k]

        scores = defaultdict(float)

        for idx in top_indices:
            sim = float(sims[idx])
            reactants = self.train_reactants[idx]

            # similarity_power > 1 bevorzugt sehr ähnliche Nachbarn stärker.
            scores[reactants] += sim ** self.similarity_power

        if len(scores) == 0:
            return self.most_common_reactants

        return max(scores.items(), key=lambda x: x[1])[0]

    def predict(self, product_smiles_list):
        return [self.predict_one(x) for x in product_smiles_list]

# %% [markdown]
# ## Zelle 7: Random Validation Split
#
# Dieser Split ist eher optimistisch.
#
# Er beantwortet:
#
# ```text
# Wie gut ist das Modell, wenn ähnliche oder gleiche Products im Training vorkommen dürfen?
# ```

# %%
def random_train_valid_split(df, valid_frac=0.2, seed=42):
    rng = np.random.default_rng(seed)

    indices = np.arange(len(df))
    rng.shuffle(indices)

    n_valid = int(len(df) * valid_frac)

    valid_idx = indices[:n_valid]
    train_idx = indices[n_valid:]

    train_df = df.iloc[train_idx].reset_index(drop=True)
    valid_df = df.iloc[valid_idx].reset_index(drop=True)

    return train_df, valid_df


train_random_df, valid_random_df = random_train_valid_split(
    df,
    valid_frac=0.2,
    seed=SEED,
)

print("Random train size:", len(train_random_df))
print("Random valid size:", len(valid_random_df))

model_random = RetrosynthesisTopKBaseline(
    radius=2,
    n_bits=2048,
    top_k=20,
    similarity_power=2.0,
    use_no_stereo_lookup=True,
)

model_random.fit(train_random_df)

valid_random_preds = model_random.predict(valid_random_df["product"].tolist())

random_acc = calculate_top1_accuracy_from_lists(
    valid_random_preds,
    valid_random_df["reactants"].tolist(),
)

print("Random validation Top-1 accuracy:", random_acc)

# %% [markdown]
# ## Zelle 8: Product-disjoint Validation Split
#
# Dieser Split ist strenger.
#
# Hier gilt:
#
# ```text
# Kein Product aus dem Validierungsset kommt im Trainingssplit vor.
# ```
#
# Dadurch testest du stärker die Generalisierung des kNN-Fallbacks.

# %%
def product_disjoint_split(df, valid_product_frac=0.2, seed=42):
    rng = np.random.default_rng(seed)

    unique_products = np.array(df["product"].unique())
    rng.shuffle(unique_products)

    n_valid_products = int(len(unique_products) * valid_product_frac)

    valid_products = set(unique_products[:n_valid_products])
    valid_mask = df["product"].isin(valid_products)

    train_df = df.loc[~valid_mask].reset_index(drop=True)
    valid_df = df.loc[valid_mask].reset_index(drop=True)

    return train_df, valid_df


train_disjoint_df, valid_disjoint_df = product_disjoint_split(
    df,
    valid_product_frac=0.2,
    seed=SEED,
)

print("Disjoint train size:", len(train_disjoint_df))
print("Disjoint valid size:", len(valid_disjoint_df))
print("Product overlap:", len(set(train_disjoint_df["product"]) & set(valid_disjoint_df["product"])))

model_disjoint = RetrosynthesisTopKBaseline(
    radius=2,
    n_bits=2048,
    top_k=20,
    similarity_power=2.0,
    use_no_stereo_lookup=True,
)

model_disjoint.fit(train_disjoint_df)

valid_disjoint_preds = model_disjoint.predict(valid_disjoint_df["product"].tolist())

disjoint_acc = calculate_top1_accuracy_from_lists(
    valid_disjoint_preds,
    valid_disjoint_df["reactants"].tolist(),
)

print("Product-disjoint validation Top-1 accuracy:", disjoint_acc)

# %% [markdown]
# ## Zelle 9: Fehleranalyse
#
# Hier schauen wir uns falsche Beispiele an.
#
# Das ist nützlich, um zu prüfen, ob:
#
# - Stereochemie häufig Probleme macht
# - sehr ähnliche Products falsche Reactants liefern
# - bestimmte Reaktionstypen systematisch scheitern

# %%
analysis_df = valid_disjoint_df.copy()
analysis_df["prediction"] = valid_disjoint_preds

analysis_df["correct"] = [
    evaluation_equal(pred, true)
    for pred, true in zip(analysis_df["prediction"], analysis_df["reactants"])
]

print("Correct:", analysis_df["correct"].sum())
print("Total:", len(analysis_df))
print("Accuracy:", analysis_df["correct"].mean())

wrong_examples = analysis_df.loc[
    ~analysis_df["correct"],
    ["product", "reactants", "prediction"],
].head(20)

wrong_examples

# %% [markdown]
# ## Zelle 10: Parameter-Suche für Top-k-Modell
#
# Diese Zelle ist die wichtigste Verbesserung gegenüber der alten Version.
#
# Sie testet:
#
# - Morgan radius
# - Fingerprint-Größe
# - Anzahl Nachbarn `top_k`
# - Stärke der Gewichtung `similarity_power`
# - Lookup ohne Stereochemie an/aus
#
# Falls die Zelle zu lange läuft, setze `FAST_SEARCH = True`.

# %%
FAST_SEARCH = True

configs = []

if FAST_SEARCH:
    # Schnellere Suche
    for radius in [2, 3]:
        for n_bits in [2048, 4096]:
            for top_k in [5, 10, 20, 50]:
                for similarity_power in [1.0, 2.0, 3.0]:
                    for use_no_stereo_lookup in [True, False]:
                        configs.append(
                            {
                                "radius": radius,
                                "n_bits": n_bits,
                                "top_k": top_k,
                                "similarity_power": similarity_power,
                                "use_no_stereo_lookup": use_no_stereo_lookup,
                            }
                        )
else:
    # Gründlichere Suche
    for radius in [1, 2, 3]:
        for n_bits in [1024, 2048, 4096]:
            for top_k in [1, 3, 5, 10, 20, 50]:
                for similarity_power in [1.0, 2.0, 3.0, 4.0]:
                    for use_no_stereo_lookup in [True, False]:
                        configs.append(
                            {
                                "radius": radius,
                                "n_bits": n_bits,
                                "top_k": top_k,
                                "similarity_power": similarity_power,
                                "use_no_stereo_lookup": use_no_stereo_lookup,
                            }
                        )

print("Number of configs:", len(configs))

results = []

for i, cfg in enumerate(configs, start=1):
    print(f"Testing {i}/{len(configs)}:", cfg)

    model = RetrosynthesisTopKBaseline(
        radius=cfg["radius"],
        n_bits=cfg["n_bits"],
        top_k=cfg["top_k"],
        similarity_power=cfg["similarity_power"],
        use_no_stereo_lookup=cfg["use_no_stereo_lookup"],
    )

    model.fit(train_disjoint_df)

    preds = model.predict(valid_disjoint_df["product"].tolist())

    acc = calculate_top1_accuracy_from_lists(
        preds,
        valid_disjoint_df["reactants"].tolist(),
    )

    results.append(
        {
            **cfg,
            "top1_accuracy": acc,
        }
    )

results_df = pd.DataFrame(results).sort_values(
    "top1_accuracy",
    ascending=False,
)

results_df.head(20)

# %% [markdown]
# ## Zelle 11: Beste Konfiguration zusätzlich auf Random Split prüfen
#
# Der private Leaderboard-Split kann eher wie Random oder eher wie Product-disjoint wirken.
#
# Deshalb ist es sinnvoll, die beste Product-disjoint-Konfiguration auch auf Random zu prüfen.

# %%
best = results_df.iloc[0].to_dict()

print("Best configuration from product-disjoint validation:")
print(best)

check_model_random = RetrosynthesisTopKBaseline(
    radius=int(best["radius"]),
    n_bits=int(best["n_bits"]),
    top_k=int(best["top_k"]),
    similarity_power=float(best["similarity_power"]),
    use_no_stereo_lookup=bool(best["use_no_stereo_lookup"]),
)

check_model_random.fit(train_random_df)

check_random_preds = check_model_random.predict(valid_random_df["product"].tolist())

check_random_acc = calculate_top1_accuracy_from_lists(
    check_random_preds,
    valid_random_df["reactants"].tolist(),
)

print("Accuracy of best disjoint config on random split:", check_random_acc)

# %% [markdown]
# ## Zelle 12: Exact-Match-Coverage im Testset prüfen
#
# Diese Zelle sagt dir, wie wichtig der Exact Lookup für das Testset wahrscheinlich ist.
#
# Wenn viele Testprodukte exakt im Training vorkommen, ist Lookup sehr wichtig.
#
# Wenn wenige exakt vorkommen, kommt fast alles vom kNN-Fallback.

# %%
train_products_set = set(df["product"])

test_products_canonical = [
    canonical_set(x, isomeric=True)
    for x in test_products_raw
]

exact_matches = sum(
    1 for p in test_products_canonical
    if p in train_products_set
)

print("Exact matches in test:", exact_matches)
print("Total test products:", len(test_products_canonical))
print("Exact match rate:", exact_matches / len(test_products_canonical))

# %% [markdown]
# ## Zelle 13: Finales Top-k-Modell auf allen Trainingsdaten trainieren
#
# Jetzt wird das finale Modell mit der besten lokalen Konfiguration auf allen Trainingsdaten trainiert.

# %%
best = results_df.iloc[0].to_dict()

print("Using final configuration:")
print(best)

final_model = RetrosynthesisTopKBaseline(
    radius=int(best["radius"]),
    n_bits=int(best["n_bits"]),
    top_k=int(best["top_k"]),
    similarity_power=float(best["similarity_power"]),
    use_no_stereo_lookup=bool(best["use_no_stereo_lookup"]),
)

final_model.fit(df)

# %% [markdown]
# ## Zelle 14: Testdaten vorhersagen

# %%
test_predictions = final_model.predict(test_products_raw)

print("Number of predictions:", len(test_predictions))

print()
print("First 10 predictions:")
for p in test_predictions[:10]:
    print(p)

# %% [markdown]
# ## Zelle 15: Submission speichern
#
# Die Submission braucht:
#
# - keinen Header
# - genau eine Spalte
# - genau so viele Zeilen wie `product_smiles_test.csv`

# %%
submission_path = Path("submission_topk.csv")

pd.Series(test_predictions).to_csv(
    submission_path,
    index=False,
    header=False,
)

print("Saved:", submission_path.resolve())
print("Rows:", len(test_predictions))

sub_check = pd.read_csv(
    submission_path,
    header=None,
    names=["prediction"],
)

print(sub_check.shape)

sub_check.head()

# %% [markdown]
# ## Zelle 16: Check gegen `sample_submission.csv`
#
# Dieser Check prüft nur die Zeilenzahl, nicht die Korrektheit.

# %%
if SAMPLE_SUB_PATH.exists():
    sample_sub = pd.read_csv(
        SAMPLE_SUB_PATH,
        header=None,
        names=["prediction"],
    )

    print("Sample submission rows:", len(sample_sub))
    print("Our submission rows:", len(test_predictions))

    if len(sample_sub) == len(test_predictions):
        print("OK: Row count matches sample_submission.csv")
    else:
        print("WARNING: Row count does not match sample_submission.csv")
else:
    print("sample_submission.csv not found, skipping check.")

# %% [markdown]
# ## Zelle 17: Optionales Ensemble
#
# Falls `submission_topk.csv` noch nicht besser genug ist, kannst du zusätzlich ein Ensemble testen.
#
# Idee:
#
# Mehrere Top-k-Modelle mit verschiedenen Parametern stimmen ab.
#
# Das kann robuster sein als eine einzelne Konfiguration.

# %%
class RetrosynthesisEnsembleBaseline:
    def __init__(self, model_configs):
        self.model_configs = model_configs
        self.models = []
        self.most_common_reactants = None

    def fit(self, train_df):
        self.models = []

        for cfg in self.model_configs:
            model = RetrosynthesisTopKBaseline(**cfg)
            model.fit(train_df)
            self.models.append(model)

        self.most_common_reactants = Counter(train_df["reactants"]).most_common(1)[0][0]

        return self

    def predict_one(self, product_smiles):
        votes = Counter()

        for model in self.models:
            pred = model.predict_one(product_smiles)
            votes[pred] += 1

        if len(votes) == 0:
            return self.most_common_reactants

        return votes.most_common(1)[0][0]

    def predict(self, product_smiles_list):
        return [self.predict_one(x) for x in product_smiles_list]


ensemble_configs = [
    {
        "radius": 1,
        "n_bits": 2048,
        "top_k": 10,
        "similarity_power": 2.0,
        "use_no_stereo_lookup": True,
    },
    {
        "radius": 2,
        "n_bits": 2048,
        "top_k": 20,
        "similarity_power": 2.0,
        "use_no_stereo_lookup": True,
    },
    {
        "radius": 3,
        "n_bits": 2048,
        "top_k": 20,
        "similarity_power": 3.0,
        "use_no_stereo_lookup": True,
    },
    {
        "radius": 2,
        "n_bits": 4096,
        "top_k": 50,
        "similarity_power": 2.0,
        "use_no_stereo_lookup": False,
    },
]

ensemble = RetrosynthesisEnsembleBaseline(ensemble_configs)
ensemble.fit(train_disjoint_df)

ensemble_preds = ensemble.predict(valid_disjoint_df["product"].tolist())

ensemble_acc = calculate_top1_accuracy_from_lists(
    ensemble_preds,
    valid_disjoint_df["reactants"].tolist(),
)

print("Ensemble product-disjoint Top-1 accuracy:", ensemble_acc)

# %% [markdown]
# ## Zelle 18: Optionale Ensemble-Submission
#
# Diese Submission kannst du zusätzlich testen:
#
# ```text
# submission_ensemble.csv
# ```

# %%
final_ensemble = RetrosynthesisEnsembleBaseline(ensemble_configs)
final_ensemble.fit(df)

ensemble_test_predictions = final_ensemble.predict(test_products_raw)

ensemble_submission_path = Path("submission_ensemble.csv")

pd.Series(ensemble_test_predictions).to_csv(
    ensemble_submission_path,
    index=False,
    header=False,
)

print("Saved:", ensemble_submission_path.resolve())
print("Rows:", len(ensemble_test_predictions))

ensemble_sub_check = pd.read_csv(
    ensemble_submission_path,
    header=None,
    names=["prediction"],
)

print(ensemble_sub_check.shape)

ensemble_sub_check.head()

# %% [markdown]
# ## Zelle 19: Optional lokale Target-Evaluation
#
# Falls du irgendwann eine lokale Target-Datei hast, z. B.:
#
# ```text
# valid_targets.csv
# ```
#
# kannst du damit direkt eine Submission auswerten.

# %%
TARGET_PATH = Path("valid_targets.csv")

if TARGET_PATH.exists():
    local_targets = pd.read_csv(
        TARGET_PATH,
        header=None,
        names=["true_reactants"],
    )

    local_preds = pd.read_csv(
        "submission_topk.csv",
        header=None,
        names=["prediction"],
    )

    acc = calculate_top1_accuracy_from_lists(
        local_preds["prediction"].tolist(),
        local_targets["true_reactants"].tolist(),
    )

    print("Local target Top-1 accuracy for submission_topk.csv:", acc)
else:
    print("No local target file found. Skipping.")

# %% [markdown]
# ## Hinweise
#
# Für deinen nächsten Upload würde ich zuerst diese Datei probieren:
#
# ```text
# submission_topk.csv
# ```
#
# Falls sie nicht besser ist, probiere danach:
#
# ```text
# submission_ensemble.csv
# ```
#
# Bei deiner bisherigen Accuracy von 0.098 kann schon eine kleine Änderung bei
# `top_k`, `similarity_power` oder `use_no_stereo_lookup` reichen, um über 0.10 zu kommen.